# Simulation Data Generation

Generate synthetic DMS data for two homologs with known mutational effects,
shifts, and global epistasis. This notebook produces the ground-truth datasets
used by all downstream simulation notebooks.

In [ ]:
config_path = "config/config.yaml"
profile = None

In [ ]:
import warnings
import os
import pickle

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import Bio.Seq
import random
import dms_variants.simulate
from dms_variants.constants import CODONS_NOSTOP, AAS_NOSTOP, AA_TO_CODONS, AAS_WITHSTOP
from plotnine import *
import multidms
from multidms.utils import split_sub

from notebooks._common import load_config, build_phenotype_fxn_dict

In [ ]:
# Load configuration
cfg = load_config(config_path, profile)
sim = cfg["simulation"]

seed = cfg["seed"]
genelength = sim["genelength"]
wt_latent = sim["wt_latent"]
stop_effect = sim["stop_effect"]
norm_weights = tuple(tuple(w) for w in sim["norm_weights"])
n_non_identical_sites = sim["n_non_identical_sites"]
min_muteffect_in_bundle = sim["min_muteffect_in_bundle"]
max_muteffect_in_bundle = sim["max_muteffect_in_bundle"]
n_shifted_non_identical_sites = sim["n_shifted_non_identical_sites"]
n_shifted_identical_sites = sim["n_shifted_identical_sites"]
shift_gauss_variance = sim["shift_gauss_variance"]
sigmoid_phenotype_scale = sim["sigmoid_phenotype_scale"]
variants_per_lib_genelength_scalar = sim["variants_per_lib_genelength_scalar"]
avgmuts = sim["avgmuts"]
bclen = sim["bclen"]
variant_error_rate = sim["variant_error_rate"]
avgdepth_per_variant = sim["avgdepth_per_variant"]
lib_uniformity = sim["lib_uniformity"]
noise = sim["noise"]
bottleneck_variants_per_lib_scalar = sim["bottleneck_variants_per_lib_scalar"]
output_dir = sim["output_dir"]
train_frac = cfg["train_frac"]

In [ ]:
%matplotlib inline

random.seed(seed)
np.random.seed(seed)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

os.makedirs(output_dir, exist_ok=True)

## Simulate reference sequence

In [ ]:
geneseq_h1 = "".join(random.choices(CODONS_NOSTOP, k=genelength))
aaseq_h1 = str(Bio.Seq.Seq(geneseq_h1).translate())
print(f"Wildtype gene of {genelength} codons:\n{geneseq_h1}")
print(f"Wildtype protein:\n{aaseq_h1}")

## Mutational effects

Simulate latent mutational effects using a `SigmoidPhenotypeSimulator`.

In [ ]:
mut_pheno_args = {
    "geneseq": geneseq_h1,
    "wt_latent": wt_latent,
    "seed": seed,
    "stop_effect": stop_effect,
    "norm_weights": norm_weights,
}

SigmoidPhenotype_h1 = dms_variants.simulate.SigmoidPhenotypeSimulator(**mut_pheno_args)
SigmoidPhenotype_h2 = dms_variants.simulate.SigmoidPhenotypeSimulator(**mut_pheno_args)

In [ ]:
p = (
    ggplot(pd.DataFrame({"muteffects": SigmoidPhenotype_h1.muteffects}), aes(x="muteffects"))
    + geom_histogram(bins=50)
    + theme_classic()
    + labs(title="Distribution of mutation effects (h1)")
)
_ = p.draw(show=True)

## Non-reference homolog sequence

Simulate a second homolog by introducing mutations at non-identical sites.

In [ ]:
non_identical_sites = sorted(random.sample(range(1, len(aaseq_h1) + 1), n_non_identical_sites))
aaseq_h2 = ""
geneseq_h2 = ""

for aa_n, aa in enumerate(aaseq_h1, 1):
    codon = geneseq_h1[(aa_n - 1) * 3 : 3 * aa_n]
    if aa_n in non_identical_sites:
        valid_bundle_mutations = [
            mut_aa
            for mut_aa in AAS_NOSTOP
            if (
                (mut_aa != aa)
                and (SigmoidPhenotype_h1.muteffects[f"{aa}{aa_n}{mut_aa}"] > min_muteffect_in_bundle)
                and (SigmoidPhenotype_h1.muteffects[f"{aa}{aa_n}{mut_aa}"] < max_muteffect_in_bundle)
            )
        ]
        assert len(valid_bundle_mutations) > 0, aa_n
        mut_aa = random.choice(valid_bundle_mutations)
        aaseq_h2 += mut_aa
        mut_codon = random.choice(AA_TO_CODONS[mut_aa])
        geneseq_h2 += mut_codon
    else:
        aaseq_h2 += aa
        geneseq_h2 += codon

homolog_seqs_df = pd.DataFrame(
    {"wt_aa_h1": list(aaseq_h1), "wt_aa_h2": list(aaseq_h2)},
    index=range(1, len(aaseq_h1) + 1),
)
homolog_seqs_df.index.name = "site"

n_diffs = len(homolog_seqs_df.query("wt_aa_h1 != wt_aa_h2"))
print("Sequence alignment of homologs h1 and h2:")
print("h1", aaseq_h1)
print("h2", aaseq_h2)
print("Number of aa differences:", n_diffs)
assert len(aaseq_h1) == len(aaseq_h2)
assert aaseq_h2 == str(Bio.Seq.Seq(geneseq_h2).translate())
assert n_diffs == len(non_identical_sites)

## Shifted mutational effects

Randomly choose sites with shifted effects between homologs.

In [ ]:
shifted_non_identical_sites = sorted(
    random.sample(non_identical_sites, n_shifted_non_identical_sites)
)
shifted_identical_sites = sorted(
    random.sample(
        list(set(range(1, len(aaseq_h1) + 1)) - set(non_identical_sites)),
        n_shifted_identical_sites,
    )
)
shifted_sites = sorted(shifted_identical_sites + shifted_non_identical_sites)
assert len(shifted_sites) == len(set(shifted_sites))
print("Sites with shifts that are...")
print(f"identical (n={len(shifted_identical_sites)}):", ", ".join(map(str, shifted_identical_sites)))
print(f"non-identical (n={len(shifted_non_identical_sites)}):", ", ".join(map(str, shifted_non_identical_sites)))

In [ ]:
def sim_mut_shift(shifted_site, mutation):
    if (not shifted_site) or ("*" in mutation):
        return 0
    else:
        return np.random.normal(loc=0, scale=shift_gauss_variance, size=1)[0]

mut_effects_df = (
    pd.DataFrame.from_dict(
        SigmoidPhenotype_h1.muteffects, orient="index", columns=["beta_h1"]
    )
    .reset_index()
    .rename(columns={"index": "mutation"})
    .assign(
        wt_aa=lambda x: x["mutation"].apply(lambda y: split_sub(y)[0]),
        site=lambda x: x["mutation"].apply(lambda y: int(split_sub(y)[1])),
        mut_aa=lambda x: x["mutation"].apply(lambda y: split_sub(y)[2]),
        shifted_site=lambda x: x["site"].isin(shifted_sites),
        shift=lambda x: x.apply(
            lambda row: sim_mut_shift(row["shifted_site"], row["mutation"]), axis=1
        ),
        beta_h2=lambda x: x["beta_h1"] + x["shift"],
    )
    .merge(homolog_seqs_df, left_on="site", right_index=True, how="left")
    .assign(bundle_mut=lambda x: x["mut_aa"] == x["wt_aa_h2"])
)

mut_effects_df[mut_effects_df["shifted_site"]][
    ["site", "wt_aa_h1", "wt_aa_h2", "mutation", "beta_h1", "shift", "beta_h2", "shifted_site"]
]

### Update h2 mutational effects

In [ ]:
assert sum(mut_effects_df["mutation"].duplicated()) == 0
for mutation in SigmoidPhenotype_h2.muteffects.keys():
    SigmoidPhenotype_h2.muteffects[mutation] = mut_effects_df.loc[
        mut_effects_df["mutation"] == mutation, "beta_h2"
    ].values[0]

wt_latent_phenotype_shift = mut_effects_df.query("bundle_mut")["beta_h2"].sum()
SigmoidPhenotype_h2.wt_latent = SigmoidPhenotype_h1.wt_latent + wt_latent_phenotype_shift

print("Characteristics of mutations separating homologs:\n")
for metric in ["beta_h1", "shift", "beta_h2"]:
    print(f"  Sum of {metric}:", round(sum(mut_effects_df.query("bundle_mut")[metric]), 2))
print("  Final WT latent phenotype of h2:", round(SigmoidPhenotype_h2.wt_latent, 2))

In [ ]:
# Fill in mutational effects for bundle sites in h2
for idx, row in mut_effects_df.query("mut_aa == wt_aa_h2").iterrows():
    for aa_mut in AAS_WITHSTOP:
        if aa_mut == row.wt_aa_h2:
            continue
        non_ref_mutation = f"{row.wt_aa_h2}{row.site}{aa_mut}"
        if aa_mut == "*":
            SigmoidPhenotype_h2.muteffects[non_ref_mutation] = stop_effect
        elif aa_mut == row.wt_aa_h1:
            SigmoidPhenotype_h2.muteffects[non_ref_mutation] = -row.beta_h2
        else:
            ref_mut = f"{row.wt_aa_h1}{row.site}{aa_mut}"
            ref_mut_effect = mut_effects_df.loc[
                mut_effects_df["mutation"] == ref_mut, "beta_h2"
            ].values[0]
            SigmoidPhenotype_h2.muteffects[non_ref_mutation] = -row.beta_h2 + ref_mut_effect

In [ ]:
mut_effects_df.to_csv(f"{output_dir}/simulated_muteffects.csv", index=False)
print(f"Saved {output_dir}/simulated_muteffects.csv ({len(mut_effects_df)} mutations)")

## Variant libraries

Simulate variant libraries for each homolog.

In [ ]:
libs = ["lib_1", "lib_2"]
variants_per_lib = variants_per_lib_genelength_scalar * genelength

CodonVariantTable_h1 = dms_variants.simulate.simulate_CodonVariantTable(
    geneseq=geneseq_h1,
    bclen=bclen,
    library_specs={lib: {"avgmuts": avgmuts, "nvariants": variants_per_lib} for lib in libs},
    seed=seed,
)
CodonVariantTable_h2 = dms_variants.simulate.simulate_CodonVariantTable(
    geneseq=geneseq_h2,
    bclen=bclen,
    library_specs={lib: {"avgmuts": avgmuts, "nvariants": variants_per_lib} for lib in libs},
    seed=seed,
)

## Variant phenotypes and enrichments

Assign latent phenotypes, observed phenotypes, and enrichments to each variant.

In [ ]:
phenotype_fxn_dict_h1 = build_phenotype_fxn_dict(
    SigmoidPhenotype_h1, sigmoid_phenotype_scale, wt_latent, is_reference=True
)
phenotype_fxn_dict_h2 = build_phenotype_fxn_dict(
    SigmoidPhenotype_h2, sigmoid_phenotype_scale, wt_latent, is_reference=False
)

# Add latent phenotypes
CodonVariantTable_h1.barcode_variant_df["latent_phenotype"] = (
    CodonVariantTable_h1.barcode_variant_df["aa_substitutions"].apply(
        phenotype_fxn_dict_h1["latentPhenotype"]
    )
)
CodonVariantTable_h2.barcode_variant_df["latent_phenotype"] = (
    CodonVariantTable_h2.barcode_variant_df["aa_substitutions"].apply(
        phenotype_fxn_dict_h2["latentPhenotype"]
    )
)

In [ ]:
# Add observed phenotypes and enrichments
for cvt, fxn_dict in [
    (CodonVariantTable_h1, phenotype_fxn_dict_h1),
    (CodonVariantTable_h2, phenotype_fxn_dict_h2),
]:
    subs = cvt.barcode_variant_df["aa_substitutions"]
    cvt.barcode_variant_df["observed_phenotype"] = subs.apply(fxn_dict["observedPhenotype"])
    cvt.barcode_variant_df["observed_enrichment"] = subs.apply(fxn_dict["observedEnrichment"])

variants_df = pd.concat([
    CodonVariantTable_h1.barcode_variant_df.assign(homolog="h1"),
    CodonVariantTable_h2.barcode_variant_df.assign(homolog="h2"),
])
print(f"Generated {len(variants_df)} variants across both homologs")

## Pre and post-selection variant read counts

In [ ]:
bottlenecks = {
    name: variants_per_lib * scalar
    for name, scalar in bottleneck_variants_per_lib_scalar.items()
}

counts_h1, counts_h2 = [
    dms_variants.simulate.simulateSampleCounts(
        variants=variants,
        phenotype_func=pheno_fxn_dict["observedEnrichment"],
        variant_error_rate=variant_error_rate,
        pre_sample={
            "total_count": variants_per_lib * np.random.poisson(avgdepth_per_variant),
            "uniformity": lib_uniformity,
        },
        pre_sample_name="pre-selection",
        post_samples={
            name: {
                "noise": noise,
                "total_count": variants_per_lib * np.random.poisson(avgdepth_per_variant),
                "bottleneck": bottleneck,
            }
            for name, bottleneck in bottlenecks.items()
        },
        seed=seed,
    )
    for variants, pheno_fxn_dict in zip(
        [CodonVariantTable_h1, CodonVariantTable_h2],
        [phenotype_fxn_dict_h1, phenotype_fxn_dict_h2],
    )
]
CodonVariantTable_h1.add_sample_counts_df(counts_h1)
CodonVariantTable_h2.add_sample_counts_df(counts_h2)

## Prep training data

Combine ground-truth phenotypes and bottleneck-derived functional scores.

In [ ]:
req_cols = ["library", "homolog", "aa_substitutions", "func_score_type", "func_score"]
ground_truth_training_set = (
    pd.concat([
        variants.barcode_variant_df[
            ["library", "aa_substitutions", "observed_phenotype", "latent_phenotype"]
        ]
        .drop_duplicates()
        .assign(homolog=homolog)
        for variants, homolog in zip(
            [CodonVariantTable_h1, CodonVariantTable_h2], ["h1", "h2"]
        )
    ])
    .melt(
        id_vars=["library", "aa_substitutions", "homolog", "latent_phenotype"],
        value_vars=["observed_phenotype"],
        var_name="func_score_type",
        value_name="func_score",
    )[req_cols]
)
print(f"Ground truth training set: {len(ground_truth_training_set)} rows")

In [ ]:
bottle_cbf = pd.concat([
    (
        variants.func_scores("pre-selection", by="aa_substitutions", libraries=libs, syn_as_wt=True)
        .assign(homolog=homolog)
        .rename({"post_sample": "func_score_type"}, axis=1)
        .astype({c: str for c in req_cols[:-1]})
    )
    for variants, homolog in zip(
        [CodonVariantTable_h1, CodonVariantTable_h2], ["h1", "h2"]
    )
])
print(f"Bottleneck functional scores: {len(bottle_cbf)} rows")

In [ ]:
def classify_variant(aa_subs):
    if "*" in aa_subs:
        return "stop"
    elif aa_subs == "":
        return "wildtype"
    elif len(aa_subs.split()) == 1:
        return "1 nonsynonymous"
    elif len(aa_subs.split()) > 1:
        return ">1 nonsynonymous"
    else:
        raise ValueError(f"unexpected aa_subs: {aa_subs}")

func_scores = (
    pd.concat([ground_truth_training_set, bottle_cbf])
    .assign(variant_class=lambda x: x["aa_substitutions"].apply(classify_variant))
    .merge(
        variants_df[["aa_substitutions", "homolog", "latent_phenotype", "library"]].drop_duplicates(),
        on=["aa_substitutions", "homolog", "library"],
        how="inner",
    )
)

func_scores["func_score_type"] = pd.Categorical(
    func_scores["func_score_type"],
    categories=["observed_phenotype", "loose_bottle", "tight_bottle"],
    ordered=True,
)
print(f"Combined functional scores: {len(func_scores)} rows")
print(f"Variant classes: {func_scores['variant_class'].value_counts().to_dict()}")

## Save outputs

In [ ]:
func_scores.to_csv(f"{output_dir}/simulated_func_scores.csv", index=False)
mut_effects_df.to_csv(f"{output_dir}/simulated_muteffects.csv", index=False)
print(f"Saved functional scores ({len(func_scores)} rows) and mutation effects ({len(mut_effects_df)} rows)")